# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

In [2]:
# Import Libraries 

import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

# Load Dataset

A = pd.read_csv("model-A.tsv", sep="\t", index_col="index").to_numpy(dtype=float)

B = pd.read_csv("model-B.tsv", sep="\t", index_col="index").to_numpy(dtype=float)

Q = pd.read_csv("cost-Q.tsv", sep="\t", index_col="index").to_numpy(dtype=float)

R = pd.read_csv("cost-R.tsv", sep="\t", index_col="index").to_numpy(dtype=float)

print("A:\n", A)
print("B:\n", B)
print("Q:\n", Q)
print("R:\n", R)

A:
 [[1. 1. 0.]
 [0. 1. 1.]
 [0. 0. 1.]]
B:
 [[0.]
 [0.]
 [1.]]
Q:
 [[5. 0. 0.]
 [0. 1. 0.]
 [0. 0. 2.]]
R:
 [[5.]]


## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [3]:
# Calculate the LQR Control Matrix
# Calculate K by solving the discrete algebraric Riccati equation

P = solve_discrete_are(A, B, Q, R)

K = np.linalg.solve(
    R + B.T @ P @ B,
    B.T @ P @ A
)

print("Intended control matrix K:")
print(K)

Intended control matrix K:
[[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [4]:
# Save the control matrix
K_df = pd.DataFrame(
    K,
    columns=["x", "v", "a"]
)

K_df.to_csv(
    "control-K-intended.tsv",
    sep="\t",
    index=False
)

K_df

,x,v,a
0,0.33446,1.304454,1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [5]:
# Load the quadcopter log
# Use the calculate control matrix to recompute the actions recorded in qc-log.tsv

log = pd.read_csv("qc-log.tsv", sep="\t")

print(log.head())

   index         x         v         a         u
0      0 -5.000000  0.000000  0.000000  1.702188
1      1 -5.000000 -0.017022  1.531969 -1.263200
2      2 -5.018724  1.452683  0.548285 -1.249321
3      3 -3.420773  1.840779 -0.521275 -0.212127
4      4 -1.395916  1.163611 -0.764317  0.452895


In [6]:
# Recalculate the control actions
states = log[["x", "v", "a"]].to_numpy()

u_check = -(states @ K.T).ravel()

print("Recomputed actions:")
print(u_check[:10])

Recomputed actions:
[ 1.67229926 -1.15209535 -1.23518258 -0.2885035   0.36920157  0.43059849
  0.18864361 -0.02480317 -0.08902456 -0.05238364]


In [7]:
# Compare the original and recomputed actions
comparison = pd.DataFrame({
    "u_actual": log["u"],
    "u_check": u_check
})

comparison["difference"] = (
    comparison["u_actual"] - comparison["u_check"]
)

print(comparison.head())

print(
    "Mean absolute difference:",
    comparison["difference"].abs().mean()
)

   u_actual   u_check  difference
0  1.702188  1.672299    0.029888
1 -1.263200 -1.152095   -0.111105
2 -1.249321 -1.235183   -0.014138
3 -0.212127 -0.288504    0.076376
4  0.452895  0.369202    0.083694
Mean absolute difference: 0.003897797050010452


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [8]:
# Save the results
qc_check = pd.DataFrame({
    "index": log["index"],
    "u_check": u_check
})

qc_check.to_csv(
    "qc-check.tsv",
    sep="\t",
    index=False
)

qc_check.head()

,index,u_check
0,0,1.672299
1,1,-1.152095
2,2,-1.235183
3,3,-0.288504
4,4,0.369202


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [9]:
# Estimate the actual control matrix
K_actual = -np.linalg.lstsq(
    states,
    log["u"].to_numpy(),
    rcond=None
)[0].reshape(1, 3)

print("Actual control matrix:")
print(K_actual)

Actual control matrix:
[[0.34043755 1.30012023 1.95011696]]


In [10]:
# Compare the intended and actual matrices
K_comparison = pd.DataFrame({
    "Variable": ["x", "v", "a"],
    "Intended K": K.flatten(),
    "Actual K": K_actual.flatten()
})

K_comparison["Difference"] = (
    K_comparison["Actual K"] -
    K_comparison["Intended K"]
)

K_comparison

,Variable,Intended K,Actual K,Difference
0,x,0.334460,0.340438,0.005978
1,v,1.304454,1.300120,-0.004334
2,a,1.858131,1.950117,0.091986


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [11]:
# Save the actual control atrix
K_actual_df = pd.DataFrame(
    K_actual,
    columns=["x", "v", "a"]
)

K_actual_df.to_csv(
    "control-K-actual.tsv",
    sep="\t",
    index=False
)

K_actual_df

,x,v,a
0,0.340438,1.30012,1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [12]:
# Load the training dataset 
train = pd.read_csv("qc-train.tsv", sep="\t")

print(train.head())
print(train.shape)

   index         x         v         a         u
0      0 -5.000000  0.000000  0.000000  1.729856
1      1 -5.000000 -0.017299  1.556871 -1.311911
2      2 -5.019028  1.476577  0.531837 -1.198850
3      3 -3.394793  1.846154 -0.493944 -0.297565
4      4 -1.364024  1.195267 -0.811147  0.472619
(100, 5)


In [13]:
# Prepare the training features
features = train[
    ["x", "v", "a", "u"]
].iloc[:-1].to_numpy()

# Dependent variables are the states at the next time step:
next_states = train[
    ["x", "v", "a"]
].iloc[1:].to_numpy()

In [14]:
# Estimate the dynamics using linear regression
coefficients = np.linalg.lstsq(
    features,
    next_states,
    rcond=None
)[0]

A_new = coefficients[:3, :].T

B_new = coefficients[3:, :].T

print("Estimated A matrix:")
print(A_new)

print("Estimated B matrix:")
print(B_new)

Estimated A matrix:
[[ 1.00000000e+00  1.10000000e+00  1.38777878e-15]
 [ 1.11768439e-16  9.00000000e-01  9.50000000e-01]
 [-3.54607141e-16 -1.05471187e-15  1.10000000e+00]]
Estimated B matrix:
[[ 2.22044605e-16]
 [-1.00000000e-02]
 [ 9.00000000e-01]]


In [15]:
# Evaluate the estimated model
predicted_states = features @ coefficients

rmse = np.sqrt(
    np.mean((next_states - predicted_states) ** 2)
)

print("Dynamics model RMSE:", rmse)

Dynamics model RMSE: 2.612420152058848e-16


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [16]:
# Save the new A matrix
A_new_df = pd.DataFrame(
    A_new,
    index=["x", "v", "a"],
    columns=["x", "v", "a"]
)

A_new_df = A_new_df.rename_axis("index")

A_new_df.to_csv(
    "model-A-new.tsv",
    sep="\t"
)

A_new_df

,x,v,a
index,,,
x,1.000000e+00,1.100000e+00,1.387779e-15
v,1.117684e-16,9.000000e-01,9.500000e-01
a,-3.546071e-16,-1.054712e-15,1.100000e+00


In [17]:
# Save the new B matrix
B_new_df = pd.DataFrame(
    B_new,
    index=["x", "v", "a"],
    columns=["u"]
)

B_new_df = B_new_df.rename_axis("index")

B_new_df.to_csv(
    "model-B-new.tsv",
    sep="\t"
)

B_new_df

,u
index,
x,2.220446e-16
v,-1.000000e-02
a,9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [18]:
acknowledgements = pd.DataFrame([
    {
        "source": "DX704 Week 7 Examples",
        "reference": "video_visualizing_linear_quadratic_regulators.ipynb",
        "usage": "Referenced LQR concepts and methods for calculating the optimal control matrix."
    },
    {
        "source": "DX704 Week 7 Examples",
        "reference": "video_simulating_a_quadcopter_with_different_cost_parameters.ipynb",
        "usage": "Referenced quadcopter simulations to understand how cost matrices Q and R influence control actions."
    },
    {
        "source": "DX704 Week 7 Examples",
        "reference": "video_quadcopters_gone_wild.ipynb",
        "usage": "Referenced quadcopter dynamics and controller behavior to understand differences between intended and actual system performance."
    },
    {
        "source": "Python Libraries",
        "reference": "NumPy, pandas, and SciPy",
        "usage": "Used for matrix calculations, solving the discrete algebraic Riccati equation, linear regression, data processing, and exporting TSV files."
    }
])

acknowledgements.to_csv(
    "acknowledgements.txt",
    sep="\t",
    index=False
)